# SHAP Global + RF y SHAP por Cluster (Secciones 10-12)
## Predicción de Deserción Estudiantil · Equipo 7

---
**Prerrequisitos:**
- `clustering/02_kmeans_independiente.ipynb` (df_pre, df_tec con clusters)
- `modelos/experimentos/random_forest.ipynb` (rf_model, feat_pre, datos de test)

Cubre:
- **§10** SHAP Global — invarianza de importancias RF entre regímenes
- **§11** RF y SHAP por Cluster (K-Means independiente)
- **§12** Análisis de invarianza SHAP PreTec21 → Tec21


## Setup e Importaciones

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

from sklearn.ensemble         import RandomForestClassifier
from sklearn.model_selection  import train_test_split
from sklearn.metrics          import roc_auc_score, recall_score, f1_score
from sklearn.impute           import SimpleImputer

try:
    import shap
    SHAP_AVAILABLE = True
    print('✓ SHAP:', shap.__version__)
except ImportError:
    SHAP_AVAILABLE = False
    print('⚠ SHAP no disponible — instalar con: pip install shap')

SEED = 42
np.random.seed(SEED)
print('✓ Librerías cargadas')

## Configuración y carga de datos

In [ ]:
PROCESSED_DIR = Path('../../data/processed')
IMG_DIR       = Path('images'); IMG_DIR.mkdir(exist_ok=True)

TARGET  = 'retention'
MIN_AUC = 0.60

import json

# ── DataFrames desde CSV ──────────────────────────────────────────────────────
df_pre   = pd.read_csv(PROCESSED_DIR / 'df_pre.csv')
df_tec   = pd.read_csv(PROCESSED_DIR / 'df_tec.csv')
df_match = pd.read_csv(PROCESSED_DIR / 'df_match.csv')

# ── Lista de features desde JSON ─────────────────────────────────────────────
with open(PROCESSED_DIR / 'feat_pre.json') as f: feat_pre = json.load(f)

# ── Modelo RF → pickle ────────────────────────────────────────────────────────
with open(PROCESSED_DIR / 'rf_model.pkl','rb') as f: rf_global = pickle.load(f)

# ── Arrays de test ────────────────────────────────────────────────────────────
X_te_pre = np.load(PROCESSED_DIR / 'X_te_pre.npy')
X_te_tec = np.load(PROCESSED_DIR / 'X_te_tec.npy')
y_te_pre = np.load(PROCESSED_DIR / 'y_te_pre.npy')
y_te_tec = np.load(PROCESSED_DIR / 'y_te_tec.npy')

FEATURE_COLS = feat_pre
print(f'✓ df_pre={df_pre.shape}  df_tec={df_tec.shape}')
print(f'✓ RF cargado  |  X_te_tec={X_te_tec.shape}')
print(f'df_match:\n{df_match.to_string(index=False)}')

## §10 — SHAP Global: Invarianza de Importancias

Compara el ranking de importancias SHAP del RF (entrenado en PreTec21)
aplicado a datos PreTec21 vs Tec21.
Correlación Spearman ≥ 0.7 → ranking invariante.

In [ ]:
if SHAP_AVAILABLE:
    def get_shap_matrix(shap_vals, class_idx=1):
        if isinstance(shap_vals, list):
            arr = np.array(shap_vals[class_idx] if len(shap_vals) > class_idx else shap_vals[0])
        else:
            arr = np.array(shap_vals)
        if arr.ndim == 3: arr = arr[:,:,class_idx]
        return arr

    explainer_rf = shap.TreeExplainer(rf_global)

    raw_pre  = explainer_rf.shap_values(X_te_pre)
    shap_pre = get_shap_matrix(raw_pre)
    mean_pre = np.abs(shap_pre).mean(axis=0)

    raw_tec  = explainer_rf.shap_values(X_te_tec)
    shap_tec = get_shap_matrix(raw_tec)
    mean_tec = np.abs(shap_tec).mean(axis=0)

    inv_df = pd.DataFrame({
        'SHAP_PreTec21': pd.Series(mean_pre, index=feat_pre).round(5),
        'SHAP_Tec21'   : pd.Series(mean_tec, index=feat_pre).round(5),
        'Rank_Pre'     : pd.Series(mean_pre, index=feat_pre).rank(ascending=False),
        'Rank_Tec'     : pd.Series(mean_tec, index=feat_pre).rank(ascending=False),
    })
    inv_df['Delta_Rank'] = (inv_df['Rank_Tec'] - inv_df['Rank_Pre']).abs()
    inv_df['Delta_Pct']  = (inv_df['Delta_Rank'] / len(inv_df) * 100).round(1)
    inv_df['Estable']    = inv_df['Delta_Pct'] < 20
    inv_df = inv_df.sort_values('Rank_Pre')

    n_stable = inv_df['Estable'].sum()
    print(f'Variables estables (Δ < 20%): {n_stable} / {len(inv_df)}')
    print()
    print(inv_df.to_string())

    rho, p = spearmanr(inv_df['Rank_Pre'].values, inv_df['Rank_Tec'].values)
    print(f'\nCorrelación Spearman PreTec21 vs Tec21: ρ = {rho:.3f}  (p = {p:.4f})')
    verdict = '✓ Alta correlación — ranking invariante' if rho >= 0.7               else '⚠ Baja correlación — ranking inestable'
    print(f'  {verdict}')

    # Scatter ranking
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(inv_df['Rank_Pre'], inv_df['Rank_Tec'], alpha=0.7, color='#2563eb', s=60)
    for feat in inv_df[~inv_df['Estable']].index:
        ax.annotate(feat, (inv_df.loc[feat,'Rank_Pre'], inv_df.loc[feat,'Rank_Tec']),
                    fontsize=7, alpha=0.8)
    ax.plot([1,len(inv_df)],[1,len(inv_df)],'k--',alpha=0.4,label='diagonal perfecta')
    ax.set_xlabel('Rank SHAP — PreTec21'); ax.set_ylabel('Rank SHAP — Tec21')
    ax.set_title(f'Invarianza SHAP Global  ρ={rho:.3f}')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMG_DIR / 'shap_invarianza_global.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Guardado: {IMG_DIR}/shap_invarianza_global.png')
else:
    print('SHAP no disponible')

## §11 — RF y SHAP por Cluster (K-Means Independiente)

Entrena un RF independiente por cluster y régimen.
Calcula SHAP top-5 por cluster para identificar los factores de riesgo locales.

In [ ]:
def train_rf_cluster_shap(df_regime, feat_cols, cluster_id, regime_name, seed=SEED):
    sub  = df_regime[df_regime['cluster']==cluster_id].copy()
    cols = [c for c in feat_cols if c in sub.columns and sub[c].std() > 0]
    if len(cols) < 3:
        print(f'  Cluster {cluster_id} [{regime_name}]: pocas features — omitido')
        return None
    X  = SimpleImputer(strategy='median').fit_transform(sub[cols].values.astype(float))
    y  = sub[TARGET].values.astype(int)
    if len(np.unique(y)) < 2 or (y==0).sum() < 5:
        print(f'  Cluster {cluster_id} [{regime_name}]: clases insuficientes — omitido')
        return None
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    rf_c = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                  class_weight='balanced', random_state=seed, n_jobs=-1)
    rf_c.fit(X_tr, y_tr)
    y_proba = rf_c.predict_proba(X_te)[:,1]
    auc_c = roc_auc_score(y_te, y_proba) if len(np.unique(y_te)) > 1 else 0.5
    rec   = recall_score(y_te, (y_proba>=0.5).astype(int), zero_division=0)
    f1    = f1_score(y_te, (y_proba>=0.5).astype(int), zero_division=0)
    shap_top = None
    if SHAP_AVAILABLE:
        exp  = shap.TreeExplainer(rf_c)
        sv   = exp.shap_values(X_te)
        if isinstance(sv, list): sv = np.array(sv[1])
        else: sv = np.array(sv)
        if sv.ndim == 3: sv = sv[:,:,1]
        mean_sv  = np.abs(sv).mean(axis=0)
        shap_top = pd.Series(mean_sv, index=cols).sort_values(ascending=False)
    fi = pd.Series(rf_c.feature_importances_, index=cols).sort_values(ascending=False)
    print(f'  Cluster {cluster_id} [{regime_name:>8}] n={len(sub):>5}  AUC={auc_c:.3f}  Recall={rec:.3f}  F1={f1:.3f}')
    return dict(cluster=cluster_id, regime=regime_name, n=len(sub),
                auc=auc_c, recall=rec, f1=f1,
                model=rf_c, feat_importance=fi, shap_top=shap_top, feat_cols=cols)

print('═══ RF por Cluster (K-Means Independiente) ═══\n')
cluster_results = []
for regime_name, df_regime in [('PreTec21', df_pre), ('Tec21', df_tec)]:
    print(f'--- {regime_name} ---')
    for k in sorted(df_regime['cluster'].unique()):
        res = train_rf_cluster_shap(df_regime, FEATURE_COLS, k, regime_name)
        if res: cluster_results.append(res)
    print()
print(f'✓ Modelos por cluster: {len(cluster_results)}')

In [ ]:
df_cm = pd.DataFrame([{k:v for k,v in r.items()
                          if k not in ('model','feat_importance','shap_top','feat_cols')}
                        for r in cluster_results]).round(3)
print('═══ Métricas por Cluster ═══')
print(df_cm.to_string(index=False))

In [ ]:
if SHAP_AVAILABLE:
    print('═══ SHAP Top-5 por Cluster ═══\n')
    for r in cluster_results:
        if r['shap_top'] is None: continue
        print(f"Cluster {r['cluster']} [{r['regime']:>8}] (n={r['n']:,}):")
        for feat, val in r['shap_top'].head(5).items():
            print(f"  {feat:<30} {val:.5f}")
        print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
K_CLUSTERS = df_pre['cluster'].nunique()

ax = axes[0]
for i, (regime_name, color) in enumerate([('PreTec21','#2563eb'),('Tec21','#dc2626')]):
    sub = df_cm[df_cm['regime']==regime_name]
    ax.bar(sub['cluster'] + i*0.35 - 0.175, sub['auc'], 0.35,
           label=regime_name, color=color, alpha=0.8)
ax.axhline(MIN_AUC, ls='--', color='gray', label=f'Min AUC={MIN_AUC}')
ax.set_xlabel('Cluster'); ax.set_ylabel('AUC-ROC')
ax.set_title('AUC-ROC por Cluster (K-Means Independiente)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
for regime_name, df_regime, color in [('PreTec21',df_pre,'#2563eb'),('Tec21',df_tec,'#dc2626')]:
    clusters = sorted(df_regime['cluster'].unique())
    drs = [(df_regime[df_regime['cluster']==c][TARGET]==0).mean() for c in clusters]
    ns  = [len(df_regime[df_regime['cluster']==c]) for c in clusters]
    offset = 0 if regime_name=='PreTec21' else 0.35
    bars = ax2.bar([c + offset for c in clusters], drs, 0.35,
                   label=regime_name, color=color, alpha=0.8)
    for bar, n in zip(bars, ns):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'n={n//1000}k', ha='center', va='bottom', fontsize=7)
ax2.set_xlabel('Cluster'); ax2.set_ylabel('Tasa de Deserción')
ax2.set_title('Tasa de Deserción por Cluster')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(IMG_DIR / 'cluster_metrics_independiente.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Guardado: {IMG_DIR}/cluster_metrics_independiente.png')

## §12 — Análisis de Invarianza SHAP (PreTec21 → Tec21)

In [ ]:
if SHAP_AVAILABLE and len(df_match) > 0:
    print('═══ Invarianza de SHAP en Clusters con Match ═══\n')
    for _, row in df_match.iterrows():
        pre_c = int(row['Pre_Cluster'])
        tec_c = int(row['Tec_Cluster'])
        quality = row['Match']

        r_pre = next((r for r in cluster_results if r['regime']=='PreTec21' and r['cluster']==pre_c), None)
        r_tec = next((r for r in cluster_results if r['regime']=='Tec21' and r['cluster']==tec_c), None)

        if r_pre is None or r_tec is None or r_pre['shap_top'] is None or r_tec['shap_top'] is None:
            print(f'  Pre_C{pre_c} → Tec_C{tec_c}: datos insuficientes')
            continue

        feats_common = list(set(r_pre['feat_cols']) & set(r_tec['feat_cols']))
        v_pre = [float(r_pre['shap_top'].get(f,0)) for f in feats_common]
        v_tec = [float(r_tec['shap_top'].get(f,0)) for f in feats_common]

        if len(feats_common) < 4:
            print(f'  Pre_C{pre_c} → Tec_C{tec_c}: pocas features comunes')
            continue

        rho_c, pval = spearmanr(v_pre, v_tec)
        print(f'Pre_C{pre_c} → Tec_C{tec_c}  |  match={quality}  |  Spearman ρ={rho_c:.3f}  (p={pval:.3f})')

        top5_pre = r_pre['shap_top'].head(5).index.tolist()
        top5_tec = r_tec['shap_top'].head(5).index.tolist()
        common_top = set(top5_pre) & set(top5_tec)
        print(f'  Top-5 PreTec21: {top5_pre}')
        print(f'  Top-5 Tec21:    {top5_tec}')
        print(f'  Features en común (top-5): {len(common_top)}/5 → {sorted(common_top)}')
        print()
else:
    print('SHAP no disponible o sin resultados de match')